In [ ]:
import pandas as pd
import json

X = pd.read_parquet("/context/data/X.parquet")
y = pd.read_parquet("/context/data/y.parquet")

with open("/context/data/moons_split.json") as f:
    splits = json.load(f)

X_train      = X[X["moon"].isin(splits["train"])]
y_train      = y[y["moon"].isin(splits["train"])]
X_test_cloud = X[X["moon"].isin(splits["reduced_cloud"])]
y_test_cloud = y[y["moon"].isin(splits["reduced_cloud"])]


In [ ]:
import joblib
import os
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from scipy.stats import pearsonr


def train(X_train, y_train, model_directory_path, loop_moon, embargo):
    feature_cols = [c for c in X_train.columns if c.startswith("Feature_")]

    df = X_train.merge(y_train[["moon", "id", "target"]], on=["moon", "id"])
    df = df.dropna(subset=["target"])

    all_moons = sorted(df["moon"].unique())

    # ── ONLY CHANGE: rolling window CV ───────────────────────────────
    CANDIDATE_WINDOWS = [50, 100, 150, 200, 300, 400]
    VAL_MOONS         = 15

    val_moon_ids    = all_moons[-VAL_MOONS:]
    avail_for_train = all_moons[:-VAL_MOONS]

    df_val = df[df["moon"].isin(val_moon_ids)]

    best_ic = -np.inf
    best_w  = CANDIDATE_WINDOWS[-1]

    print("  Rolling window CV:")

    for w in CANDIDATE_WINDOWS:

        tr_moons = avail_for_train[-w:]

        df_tr = df[df["moon"].isin(tr_moons)]

        if len(df_tr) < 50:
            continue

        model = LGBMRegressor(
            n_estimators=200,
            learning_rate=0.05,
            random_state=42,
            verbose=-1
        )

        model.fit(
            df_tr[feature_cols].fillna(0),
            df_tr["target"]
        )

        corrs = []

        for moon in val_moon_ids:

            grp = df_val[df_val["moon"] == moon]

            if len(grp) < 2:
                continue

            r, _ = pearsonr(
                model.predict(grp[feature_cols].fillna(0)),
                grp["target"]
            )

            if not np.isnan(r):
                corrs.append(r)

        ic = float(np.mean(corrs)) if corrs else -np.inf

        print(f"    window={w:4d} moons  val IC={ic:+.5f}")

        if ic > best_ic:
            best_ic = ic
            best_w  = w

    print(f"  Best window = {best_w} moons  (val IC={best_ic:+.5f})")

    fit_moons = all_moons[-best_w:]

    df = df[df["moon"].isin(fit_moons)]

    print(
        f"  Refit on {len(fit_moons)} moons "
        f"({fit_moons[0]}→{fit_moons[-1]}), {len(df):,} rows"
    )

    X = df[feature_cols].fillna(0)
    y = df["target"]

    model = LGBMRegressor(
        n_estimators=200,
        learning_rate=0.05,
        random_state=42
    )

    model.fit(X, y)

    joblib.dump(model, os.path.join(model_directory_path, "model.joblib"))

In [ ]:
def infer(X_test, model_directory_path, loop_moon, embargo):

    model = joblib.load(
        os.path.join(model_directory_path, "model.joblib")
    )

    feature_cols = [
        c for c in X_test.columns
        if c.startswith("Feature_")
    ]

    preds = model.predict(
        X_test[feature_cols].fillna(0)
    )

    return pd.DataFrame({
        "moon":       X_test["moon"].values,
        "id":         X_test["id"].values,
        "prediction": preds,
    })

In [ ]:
import os
import numpy as np
from scipy.stats import pearsonr

embargo   = 4
model_dir = "./model"

os.makedirs(model_dir, exist_ok=True)

train(X_train, y_train, model_dir, loop_moon=0, embargo=embargo)

results = []

for moon in splits["reduced_cloud"]:

    pred = infer(
        X_test_cloud[X_test_cloud["moon"] == moon],
        model_dir,
        loop_moon=moon,
        embargo=embargo
    )

    results.append(pred)

predictions = pd.concat(results)

corrs = []

for moon in splits["reduced_cloud"]:

    merged = predictions[predictions["moon"] == moon].merge(
        y_test_cloud[y_test_cloud["moon"] == moon],
        on=["moon", "id"]
    )

    if len(merged) > 1:
        r, _ = pearsonr(
            merged["prediction"],
            merged["target"]
        )

        corrs.append(r)

        print(f"Moon {moon}: Pearson r = {r:.4f}")

print(f"\nMean IC = {np.mean(corrs):.4f}")